In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
%run /Workspace/Users/jilmedavidesco@gmail.com/pipeline_project/1_code/1_setup/2_utilities

In [0]:
dbutils.widgets.text("catalog", "atlone_enterprise", "Catalog")
dbutils.widgets.text("data_source", "orders", "Data Source")

catalog = dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("data_source")

In [0]:

df_parent = spark.table(f"{catalog}.{gold_schema}.parent_fact_{data_source}")
df_parent.show(3)

In [0]:

# 2. Convertir la fecha al primer día del mes y sumar las cantidades
df_parent_mensual = (df_parent
    .withColumn("date", F.trunc("date", "MM"))
    .groupBy("date", "product_code", "customer_code")
    .agg(F.sum("sold_quantity").alias("sold_quantity"))
)

print("before: ", df_parent.count())
print("after: ", df_parent_mensual.count())


In [0]:
df_parent_mensual.write\
    .format("delta")\
    .option("delta.enableChangeDataFeed", "true")\
    .mode("overwrite")\
    .saveAsTable(f"{catalog}.{gold_schema}.4_fact_{data_source}")

In [0]:
df_child = spark.sql(f"SELECT date, product_code, customer_code, sold_quantity FROM {catalog}.{gold_schema}.child_fact_{data_source}")
df_child.show(3)

In [0]:
# Convertir a mensual y sumar cantidades
df_child_mensual = (df_child
    .withColumn("date", F.trunc("date", "MM"))
    .groupBy("date", "product_code", "customer_code")
    .agg(F.sum("sold_quantity").alias("sold_quantity"))
)

print("before: ", df_child.count())
print("after: ", df_child_mensual.count())


delta_table = DeltaTable.forName(spark, "atlone_enterprise.gold.4_fact_orders")

delta_table.alias("target").merge(
    source=df_child_mensual.alias("source"),
    condition="target.date = source.date AND target.product_code = source.product_code AND target.customer_code = source.customer_code"
).whenMatchedUpdate(
    set={
        "sold_quantity": "target.sold_quantity + source.sold_quantity"
    }
).whenNotMatchedInsert(
    values={
        "date": "source.date",
        "product_code": "source.product_code",
        "customer_code": "source.customer_code",
        "sold_quantity": "source.sold_quantity"
    }
).execute()

print("Consolidación mensual de Hechos completada exitosamente.")